# 4주차 — Model I/O (1) 로컬 LLM과 상용 LLM을 하나의 인터페이스로 (Colab판)

「최신인공지능」 2026 · 4주차 실습 · 2026년 9월 18일 (금)

> **"코드 한 줄만 바꿉니다. 나머지는 그대로입니다."**
> 3주차에 만든 체인에서 **모델 자리만** 교체합니다.

| 실습 | 교시 | 내용 |
|------|------|------|
| 확인 B | 1교시 | VRAM 요구량 어림 계산 |
| 실습 0 ★ | 1교시 | Modelfile로 나만의 챗봇 — 코드 설정이 이긴다 |
| — | 2교시 | `.bind()` 틀려 보기 · 공급자별 이름 — 조용히 무시되는 것 실측 |
| 실습 1 ★★ | 2교시 | **① 내 체인에서 `llm =` 한 줄 교체 · ② 과제 3종 × 모델 2종 측정** |
| 실습 2 ★ | 2교시 | 6기준 비교표 작성 (정답 확인 · 로딩 뺀 속도 · 한 달 비용) |
| — | 3교시 | `temperature` · `max_tokens` · `timeout` · `max_retries` |
| 실습 3 | 3교시 | 상용 실패 → 로컬 대체 (`with_fallbacks`) |
| 실습 4 ★ | 3교시 | 첫 토큰 지연 측정 → 비교표 완성 |

> ### ⚠️ Colab 과 실습실의 차이 — 1교시에 반드시 짚을 것
>
> 이 차시는 **"내 하드웨어 기준으로 판단한다"** 가 주제입니다.
> 그런데 **Colab T4 의 VRAM 은 15GB, 실습실 PC 는 8GB** 입니다.
>
> | | 실습실 PC | Colab T4 |
> |---|---|---|
> | VRAM | **8GB** | 15GB |
> | 12B 모델 | 경계~초과 | 여유 |
> | `ollama ps` 의 PROCESSOR | GPU | GPU (또는 GPU 미배정 시 100% CPU) |
>
> 아래 VRAM 계산 셀은 **두 기준을 나란히** 출력합니다.
> **비교표에 적을 값은 "내가 배포할 환경"(=8GB) 기준입니다.**

## 0. 환경 준비

In [ ]:
# ══════════════════════════════════════════════════════════════
#  Colab 환경 준비 — 매 세션 1회 실행 (재실행 안전)
# ══════════════════════════════════════════════════════════════
WEEK_MODELS   = ["chat", "small"]             # 실습 1 이 2종 비교라 소형도 받습니다
WEEK_PACKAGES = "langchain langchain-core langchain-ollama langchain-openai python-dotenv"
WEEK_SECRETS  = ["OPENAI_API_KEY"]            # 없으면 자동으로 로컬 2종 비교로 전환됩니다

# ──────────────────────────────────────────────────────────────
import os, shutil, subprocess, sys, time, urllib.request

IN_COLAB = "google.colab" in sys.modules
def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

GPU   = shutil.which("nvidia-smi") is not None and sh("nvidia-smi").returncode == 0
CHAT  = os.environ.setdefault("MODEL",       "gemma3:4b" if GPU else "gemma3:1b")
SMALL = os.environ.setdefault("SMALL_MODEL", "gemma3:1b")
EMBED = os.environ.setdefault("EMBED_MODEL", "nomic-embed-text")
TOOL  = os.environ.setdefault("TOOL_MODEL",  "qwen3:4b")
PICK  = {"chat": CHAT, "small": SMALL, "embed": EMBED, "tool": TOOL}

print(f"[1/5] 런타임   {'GPU 있음 ✅' if GPU else 'CPU 전용 ⚠️'}   →  대화 모델 {CHAT}")
if not GPU:
    print("       [런타임] > [런타임 유형 변경] > T4 GPU 로 바꾸면 4b 모델을 쓸 수 있습니다.")

print("[2/5] 패키지 설치 중…")
r = sh(f"{sys.executable} -m pip install -q {WEEK_PACKAGES}")
print("       ✅ 완료" if r.returncode == 0 else "       ❌ 실패\n" + r.stderr[-600:])

if shutil.which("ollama") is None:
    print("[3/5] Ollama 설치 중… (약 30초)")
    sh("curl -fsSL https://ollama.com/install.sh | sh")
print("[3/5] Ollama  " + ("✅ 준비됨" if shutil.which("ollama") else "❌ 설치 실패"))

def alive():
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        return True
    except Exception:
        return False

if not alive():
    subprocess.Popen(["ollama", "serve"],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(60):
        if alive():
            break
        time.sleep(1)
print("[4/5] 서버    " + ("✅ 응답함" if alive() else "❌ 미응답 — 이 셀을 다시 실행하세요"))

have = {ln.split()[0] for ln in sh("ollama list").stdout.splitlines()[1:] if ln.strip()}
for key in WEEK_MODELS:
    name = PICK[key]
    if name in have:
        print(f"[5/5] {name:<20s} ✅ 이미 있음")
        continue
    print(f"[5/5] {name:<20s} ⏳ 내려받는 중… (진행 표시 없이 수 분 걸립니다)")
    t0 = time.time()
    r = sh(f"ollama pull {name}")
    print(f"       {'✅ 완료' if r.returncode == 0 else '❌ 실패'}  ({time.time() - t0:.0f}초)")
    if r.returncode != 0:
        print(r.stderr[-400:])

for k in WEEK_SECRETS:
    if not os.getenv(k) and IN_COLAB:
        try:
            from google.colab import userdata
            os.environ[k] = userdata.get(k)
        except Exception:
            pass
    print(f"[키]  {k:<20s} " + ("✅ 설정됨" if os.getenv(k) else "⬜ 없음 (없어도 진행됩니다)"))

print("\n" + "=" * 62)
print(f"준비 완료 — MODEL='{CHAT}'  SMALL_MODEL='{SMALL}'")
print("=" * 62)

## 1교시 — 모델을 조회하고, 내 하드웨어에 올라가는지 판단한다

실습실에서는 터미널에서 직접 칩니다. Colab에서는 `!` 로 같은 명령을 실행합니다.

In [ ]:
# 받아 놓은 모델 목록
!ollama list

In [ ]:
# ★ 파라미터 수 · 컨텍스트 길이 · 양자화 조회 — 아래 계산의 입력값이 여기 있습니다
import os
!ollama show {os.environ["MODEL"]}

### 확인 B — VRAM 요구량 어림 계산

**손으로 먼저 계산하고, 그 다음 이 셀로 답을 맞춰 봅니다.**

```
필요 VRAM = ① 가중치 + ② KV 캐시 + ③ 실행 오버헤드

① 가중치(GB) = 파라미터 수(B) × 실효 비트수 ÷ 8
                Q4_K_M 의 실효 비트는 4가 아니라 약 4.5    ★
② KV 캐시     = 파라미터 1B · 컨텍스트 1K 당 대략 12~13MB
③ 오버헤드    = CUDA 컨텍스트·런타임으로 0.5 ~ 1GB
```

⚠️ 전부 어림값입니다. 정확한 값이 목적이 아니라
**"올라가는가 / 아슬아슬한가 / 안 되는가"를 3초 안에 가르는 것**이 목적입니다.

In [ ]:
import subprocess

# ── 실습실 PC 기준 (비교표에 적을 값은 이쪽입니다) ★ ──
LAB_VRAM_GB = 8.0

# ── 지금 이 Colab 런타임의 실제 VRAM ──
def detect_vram() -> float | None:
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=10,
        )
        return int(out.stdout.strip().splitlines()[0]) / 1024
    except Exception:
        return None

COLAB_VRAM_GB = detect_vram()

EFFECTIVE_BITS   = 4.5      # Q4_K_M 의 실효 비트수 — 중요한 부분에 더 높은 비트를 섞으므로 4가 아니다 ★
KV_GB_PER_B_PER_K = 0.0125  # 파라미터 1B · 컨텍스트 1K 당 KV 캐시 (GB) 어림값
OVERHEAD_GB      = 0.7      # CUDA 컨텍스트 등 실행 오버헤드


def estimate(params_b: float, ctx_tokens: int, bits: float = EFFECTIVE_BITS) -> dict:
    """파라미터 수(B)와 컨텍스트 길이(토큰)로 필요 VRAM을 어림한다."""
    weights = params_b * bits / 8                                    # ① 가중치
    kv      = params_b * (ctx_tokens / 1024) * KV_GB_PER_B_PER_K     # ② KV 캐시
    return {"weights": weights, "kv": kv, "overhead": OVERHEAD_GB,
            "total": weights + kv + OVERHEAD_GB}


def verdict(total_gb: float, vram: float) -> str:
    """주어진 VRAM 에서 어떻게 되는가 — 세 갈래로만 가른다."""
    if total_gb <= vram * 0.8:
        return "[가능] 여유 있음"
    if total_gb <= vram * 1.1:
        # 어림값이므로 VRAM 근처(±10%)는 '경계'로 본다.
        # 에러가 아니라 '느려지는' 구간이라 이게 더 위험하다 ★
        return "[경계] 넘치기 쉬움 - 일부 레이어가 CPU로 밀릴 수 있음"
    return "[초과] CPU 분산으로 수 배 느려짐"


def table(ctx: int) -> None:
    print(f"── 컨텍스트 {ctx // 1024}K 기준 " + "─" * 40)
    for label, params_b in [("4B", 4.0), ("8B", 8.0), ("12B", 12.0)]:
        r = estimate(params_b, ctx)
        line = (f"  {label:<5} 가중치 {r['weights']:>5.1f}GB  KV {r['kv']:>4.1f}GB  "
                f"오버헤드 {r['overhead']:>4.1f}GB  = 합계 {r['total']:>5.1f}GB")
        print(line)
        print(f"        └ 실습실 8GB   {verdict(r['total'], LAB_VRAM_GB)}   ★ 비교표에는 이 판정을")
        if COLAB_VRAM_GB:
            print(f"        └ Colab {COLAB_VRAM_GB:.0f}GB  {verdict(r['total'], COLAB_VRAM_GB)}")


print(f"양자화 Q4_K_M (실효 {EFFECTIVE_BITS}비트)")
print(f"실습실 VRAM {LAB_VRAM_GB:.0f}GB / 지금 런타임 "
      + (f"{COLAB_VRAM_GB:.0f}GB" if COLAB_VRAM_GB else "GPU 없음"))
print()
table(4096)
print()
table(8192)      # ← 같은 모델, 컨텍스트만 8K 로 늘리면 (num_ctx 의 대가) ★

print()
print("[정리] 실습실 8GB 기준에서 12B는 '되긴 하는데 느린' 구간입니다.")
print("   인터넷 가이드가 권하는 모델이라도, 판단은 내 하드웨어 기준으로.")
print("   ⚠️ Colab 15GB 에서 여유롭다고 실습실에서도 여유로운 것이 아닙니다. ★")

In [ ]:
# ── 내 화면의 값으로 계산하기 ──
# 파라미터 수는 `ollama show` 의 parameters, 컨텍스트는 `ollama ps` 의 CONTEXT(실제 창)를 넣으십시오.
# ⚠️ `ollama show` 의 context length(131072)는 최대치입니다 — 넣으면 틀린 결론이 나옵니다.
PARAMS_B = 4.3
CTX      = 4096

r = estimate(PARAMS_B, CTX)
print(f"파라미터 {PARAMS_B}B / 컨텍스트 {CTX}")
print(f"  가중치 {r['weights']:.1f}GB + KV {r['kv']:.1f}GB + 오버헤드 {r['overhead']:.1f}GB "
      f"= {r['total']:.1f}GB")
print(f"  실습실 8GB : {verdict(r['total'], LAB_VRAM_GB)}")

In [ ]:
# 모델을 메모리에 올리고 실제 점유를 확인한다
import os
!ollama run {os.environ["MODEL"]} "안녕" > /dev/null 2>&1
print("── ollama ps — SIZE 와 PROCESSOR 열을 보세요 ★ ──")
!ollama ps

> **`ollama ps` 에서 볼 것**
>
> | 열 | 의미 |
> |---|---|
> | `SIZE` | 실제 메모리 점유 — 위 계산값과 대조하십시오 ★ |
> | `PROCESSOR` | `100% GPU` 면 정상. `xx% CPU` 가 섞이면 VRAM 초과로 밀린 것 |
>
> GPU 런타임이 아니면 `100% CPU` 로 나옵니다. 그 상태로도 실습은 되지만 매우 느립니다.

## 실습 0 (1교시 §3) — Modelfile로 나만의 챗봇

실습실에서는 `code/Modelfile` 을 VS Code 로 열고 터미널에서 `ollama create` 합니다.
Colab에서는 셀에서 파일을 만들어 **같은 명령**을 실행합니다.

| 지시어 | 하는 일 | 오늘 |
|---|---|---|
| `FROM` | 어떤 모델에서 출발하나 — 필수 · 맨 위 | ★ |
| `SYSTEM` | 모든 대화 앞에 붙는 역할 지시 | ★ |
| `PARAMETER` | 기본 설정값 (`temperature` · `num_ctx` …) | ★ |
| `MESSAGE` | 예시 대화 (user / assistant) | 씀 |
| `TEMPLATE` · `ADAPTER` | 대화 서식 · LoRA | ✕ |

> ⚠️ **Modelfile 은 모델을 다시 학습시키지 않습니다.** 가중치는 그대로이고,
> **앞에 붙일 글과 기본값**만 저장합니다. 그래서 `create` 는 1초, 디스크는 약 1KB 입니다.

In [ ]:
# 원본 모델의 레시피 카드 — FROM · TEMPLATE · PARAMETER · LICENSE
import os
!ollama show {os.environ["MODEL"]} --modelfile | grep -E '^(FROM|PARAMETER|TEMPLATE|LICENSE|SYSTEM)'

In [ ]:
# ── 실습 0-① Modelfile 만들기 ──────────────────────────
#    FROM 은 이 런타임의 대화 모델을 씁니다 (GPU 면 gemma3:4b, CPU 면 gemma3:1b)
import os, pathlib

MODELFILE = f'''FROM {os.environ["MODEL"]}

SYSTEM """
당신은 '파이봇'입니다. 파이썬을 처음 배우는 대학생을 돕는 튜터입니다.
- 한국어로, 다섯 문장 이내로 답합니다.
- 코드는 10줄 이내 예제 하나만 보여 줍니다.
- 과제 정답을 통째로 주지 말고, 먼저 힌트를 줍니다.
"""

PARAMETER temperature 0.3
PARAMETER num_ctx 4096
PARAMETER num_predict 400

MESSAGE user 변수가 뭐예요?
MESSAGE assistant 변수는 값에 붙이는 이름표예요. age = 20 이라고 쓰면 20에 age라는 이름표가 붙고, 이후 age를 부르면 20이 나옵니다. 직접 name 변수를 하나 만들어 볼까요?
'''
pathlib.Path("Modelfile").write_text(MODELFILE, encoding="utf-8")   # ★ UTF-8 로 저장
print(MODELFILE)

In [ ]:
# 만들고 → 확인하고 → 물어본다
!ollama create py-tutor -f Modelfile 2>/dev/null && echo "✅ create 완료"
!ollama show py-tutor
!ollama run py-tutor "리스트가 뭐야?" 2>/dev/null

### 실습 0-② 나만의 챗봇으로 바꾸기

위 셀의 `SYSTEM` · `PARAMETER` · `MESSAGE` 를 고치고, 이름을 바꿔 다시 만드세요.

| 항목 | 조건 |
|---|---|
| 이름 | 영어 소문자 · 숫자 · 하이픈 (`interview-bot`) — 한글 이름은 `invalid model name` |
| `SYSTEM` | 역할 + **규칙 세 개 이상** (말투 · 길이 · 금지) |
| `PARAMETER` | `temperature` 를 역할에 맞게 |
| `MESSAGE` | 예시 대화 한 쌍 이상 |

```
!ollama create my-bot -f Modelfile
!ollama run my-bot "질문"
```

In [ ]:
# ── 실습 0-③ 코드에서 부르면 — 누가 이기나 ★★ ─────────────
#    답의 '내용'이 아니라 '토큰 수'를 봅니다 (code/my_bot.py 와 같은 실험)
import os
from langchain_ollama import ChatOllama

BASE, BOT = os.environ["MODEL"], "py-tutor"
Q = [("human", "리스트가 뭐야?")]
S = [("system", "당신은 친절한 비서입니다."), ("human", "리스트가 뭐야?")]

def tokens(model, messages, **params):
    r = ChatOllama(model=model, **params).invoke(messages)
    u = r.usage_metadata or {}
    return u.get("input_tokens"), u.get("output_tokens"), r.content

b1, _, _ = tokens(BASE, Q, num_predict=1)
p1, _, _ = tokens(BOT, Q, num_predict=1)
b2, _, _ = tokens(BASE, S, num_predict=1)
p2, _, _ = tokens(BOT, S, num_predict=1)
_, out3, ans3 = tokens(BOT, Q, num_predict=20)

print(f"① 그냥 부르기        입력 토큰  {BASE} {b1:>4}  →  {BOT} {p1:>4}   (+{p1 - b1} = SYSTEM + MESSAGE)")
print(f"② 코드에서 system    입력 토큰  {BASE} {b2:>4}  →  {BOT} {p2:>4}   (+{p2 - b2} = MESSAGE 만 → SYSTEM 교체됨)")
print(f"③ 코드 num_predict=20 출력 토큰 {out3}  → Modelfile 의 400 을 덮어씀")
print("   ", ans3)

> **Modelfile 은 '기본값'입니다. 코드에서 주면 코드가 이깁니다.**
>
> | 코드에서 준 것 | Modelfile 쪽은 |
> |---|---|
> | `system` 메시지 | `SYSTEM` 이 빠짐 — **교체** (MESSAGE 예시는 남음) |
> | `num_predict` · `temperature` 등 | `PARAMETER` 를 **덮어씀** |
>
> 2교시에 `ChatOpenAI` 로 바꾸면 파이봇의 설정은 **따라가지 않습니다** → 모델을 갈아끼울 코드라면 **설정은 코드에**.
>
> 자료 제작 PC 실측(gemma3:4b · Ollama 0.34.0): ① 15 → 172 · ② 31 → 100 · ③ 20.
> Colab 의 모델이 1b 라면 숫자는 달라도 **차이의 방향**은 같습니다.

## 2교시 2절 — `.bind()` · 틀려 보기 · 조용히 무시되는 이름

> 1절 **`.env` 키 넣기**는 Colab 에서는 **0절 환경 준비 셀**이 대신합니다 (좌측 🔑 보안 비밀의 `OPENAI_API_KEY`).
> 아래 ③·⑤ 는 OpenAI 쪽 에러·경고를 보여 주므로 **키가 있어야** 실행됩니다. 없으면 «건너뜀» 으로 나옵니다.

생성자에 설정을 주는 방법(**방법 A**)은 1교시 실습 0-③(`my_bot.py`)에서 이미 해 봤습니다 — `ChatOllama(model="py-tutor", num_predict=20)`, 코드에서 주면 코드가 이깁니다.
오늘은 두 가지를 새로 봅니다.

- **B) `.bind()` 로 덧붙이기** — 모델 객체 하나로 설정만 다른 체인을 파생시킬 때 ★
  (같은 방식이 9주차 도구 호출 `bind_tools` 에서 다시 나옵니다)
- ★ **공급자를 바꾸면 이름이 달라지고, 어떤 이름은 에러 없이 무시됩니다**

> ### ⚠️ 함정 — 같은 `.bind()` 인데 공급자마다 받는 모양이 반대입니다 ★★
>
> **공급자** = 모델을 실제로 돌려 주는 쪽. 오늘은 둘 — **Ollama**(이 노트북에서 띄운 서버) · **OpenAI**(상용 API).
> `.bind()` 는 값을 **각 회사 라이브러리에 그대로 넘기므로**, 받는 모양이 다르면 결과가 반대가 됩니다.
>
> | | Ollama — `ChatOllama` | OpenAI — `ChatOpenAI` |
> |---|---|---|
> | `.bind()` 값을 넘겨받는 함수 | `ollama` 의 `Client.chat()` | `openai` 의 `chat.completions.create()` |
> | `temperature` 를 받는 자리 | `options={...}` 묶음 **안쪽** | **맨 바깥** 인자 |
> | `bind(temperature=0.9)` | ❌ `TypeError` | ✅ 됨 |
> | `bind(options={"temperature": 0.9})` | ✅ 됨 | ❌ `TypeError` |
>
> ```python
> base.bind(temperature=0.9)                 # Ollama → TypeError 로 죽습니다
> base.bind(options={"temperature": 0.9})    # Ollama 는 이렇게 써야 합니다
> ```

| 구역 | 보여 주는 것 (`code/bind_params.py` 와 같은 실험) | 실행 |
|---|---|---|
| ② | `.bind(options={...})` — Ollama 에서 되는 형태. **틀려 보기**: 표시한 줄을 고쳐 실행 → `TypeError` | 항상 |
| ③ | OpenAI 에 `options=` — 반대로 `TypeError` (요청 전이라 비용 0원) | 키 있을 때 |
| ④ | `ChatOllama(max_tokens=10)` — **에러 없이 무시** / `num_predict=10` 은 10에서 잘림(`length`) ★★ | 항상 |
| ⑤ | `ChatOpenAI(num_predict=300)` — 경고 후 `model_kwargs` 로 옮겨져 요청에 그대로 실림 | 키 있을 때 |
| ⑥ | 이름 확인 — 그 클래스가 아는 이름인가 (`model_fields`) | 항상 |

In [ ]:
# ── 2교시 2절 · code/bind_params.py 와 같은 실험 ─────────────
import os
import warnings
from langchain_ollama import ChatOllama
from langchain_openai import ChatOpenAI

MODEL = os.environ["MODEL"]
OPENAI_MODEL = "gpt-4o-mini"          # 🔶 수업 전날 공식 문서에서 확인해 확정할 것
HAS_KEY = bool(os.getenv("OPENAI_API_KEY"))

QUESTION = "파이썬의 장점 3가지를 각각 한 문장으로 알려줘."


def title(text: str) -> None:
    print()
    print("=" * 60)
    print(text)
    print("=" * 60)


# ── ① 방법 A · 1교시 복습 — 생성자에 지정 (여기서는 호출하지 않는다) ──
print("[방법 A · 1교시 복습] 생성자에 지정 - ChatOllama(model=..., temperature=0.2, num_predict=300)")
print("                     실습 0-③(my_bot.py) 에서 해 봤으므로 여기서는 호출하지 않습니다.")

# ── ② 방법 B: .bind() — 모델 객체는 하나, 설정만 덧붙인다 ──
title("② .bind() - 모델 객체는 하나, 설정만 덧붙인다")
base = ChatOllama(model=MODEL)

# ⚠️ ChatOllama 에서는 options={...} 로 감싸야 합니다.  ★★
#    틀려 보기: 아래 줄을  short = base.bind(temperature=0.9)  로 고쳐 실행해 보세요.
#    → .bind() 줄이 아니라 invoke 할 때 TypeError 가 납니다. (확인했으면 되돌리기)
#
# ⚠️ options 는 '덧붙이기'가 아니라 '통째 교체'입니다.
#    생성자에서 준 num_predict · num_ctx 를 유지하려면 여기 같이 적어야 합니다.
short = base.bind(options={"temperature": 0.2, "num_predict": 60})

answer = short.invoke(QUESTION).content.replace("\n", " ")
print(f"  {answer[:90]}...")

In [ ]:
# ── ③ OpenAI 에 Ollama 형태(options=)로 주면 — 반대로 TypeError (키가 있을 때만) ──
title("③ OpenAI 에 Ollama 형태(options=)로 주면 - 반대로 TypeError")
if not HAS_KEY:
    print("  [건너뜀] OPENAI_API_KEY 가 없습니다.")
    print("           (ChatOpenAI 는 키가 없으면 객체를 만들 때부터 막힙니다)")
else:
    try:
        ChatOpenAI(model=OPENAI_MODEL).bind(options={"temperature": 0.9}).invoke("hi")
    except TypeError as e:
        print(f"  TypeError: {e}")
        print("  → 요청을 보내기 전에 파이썬에서 막힙니다. 비용은 0원입니다.")

# ── ④ 조용히 무시되는 이름 ★★ ──────────────────────────────
title("④ 조용히 무시되는 이름 - ChatOllama 에 OpenAI 이름(max_tokens)을 주면")
cases = [
    ("max_tokens=10  (OpenAI 이름)", ChatOllama(model=MODEL, max_tokens=10)),
    ("num_predict=10 (Ollama 이름)", ChatOllama(model=MODEL, num_predict=10)),
]
for label, llm in cases:
    msg = llm.invoke(QUESTION)
    out = (msg.usage_metadata or {}).get("output_tokens")
    reason = msg.response_metadata.get("done_reason")
    print(f"  {label} → 출력 {out} 토큰 · done_reason={reason}")

print("  → 두 줄 모두 에러가 없었습니다. 10에서 잘린 것(length)은 num_predict 쪽뿐입니다.")
print("    에러가 나면 다행이고, 조용히 무시되면 위험합니다.")

# ── ⑤ OpenAI 에 Ollama 이름(num_predict)을 주면 (키가 있을 때만) ──
title("⑤ OpenAI 에 Ollama 이름(num_predict)을 주면 - 경고 후 요청에 그대로 실린다")
if not HAS_KEY:
    print("  [건너뜀] OPENAI_API_KEY 가 없습니다.")
else:
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        llm = ChatOpenAI(model=OPENAI_MODEL, num_predict=300)  # 만들기만 — 호출하지 않는다

    for w in caught:
        print(f"  [경고] {' '.join(str(w.message).split())}")
    print(f"  model_kwargs = {llm.model_kwargs}  ← 이 값이 요청 본문에 그대로 실립니다")
    print("  → 조용히 무시되지는 않지만 OpenAI 가 모르는 이름입니다. OpenAI 이름은 max_tokens")

# ── ⑥ 이름 확인 ──────────────────────────────────────────
title("⑥ 이름 확인 - 그 클래스가 아는 이름인가 (model_fields)")
for cls in (ChatOllama, ChatOpenAI):
    for name in ("num_predict", "max_tokens"):
        known = "있음" if name in cls.model_fields else "없음"
        print(f"  {cls.__name__:<11} '{name}' : {known}")

print()
print("[정리] temperature 는 공통, 최대 길이는 Ollama num_predict / OpenAI max_tokens.")
print("       .bind() 도 공급자마다 받는 모양이 반대입니다.")
print("       LangChain 이 '모든 것'을 통일해 주지는 않습니다.")
print("       3교시 1절은 이 표에 timeout · max_retries 두 줄만 더합니다.")

## 실습 1 ★★ (2교시) — 같은 체인에 모델만 갈아끼우고 측정한다

```
chain = prompt | llm | parser
                  ▲
                  │
           여기만 바꾼다
    ┌─────────────┴─────────────┐
ChatOllama(로컬)          ChatOpenAI(상용)

prompt 그대로 · parser 그대로 · invoke 그대로     ← 오늘의 핵심 ★
```

### 실습 1-① 내 체인에서 한 줄 바꾸기 — 바로 아래 셀

실습실에서는 3주차 파일을 복사해 **`llm =` 줄 하나만** 여러분 손으로 고칩니다.

```powershell
copy ..\week03\first_chain.py first_chain.py
python first_chain.py
```

```python
from langchain_openai import ChatOpenAI     # ← import 추가
from dotenv import load_dotenv              # ← import 추가
load_dotenv()                               # ← .env 에서 키를 읽는다

# llm = ChatOllama(model=MODEL)             # 3주차의 그 줄
llm = ChatOpenAI(model="gpt-4o-mini")       # ← 바꾼 한 줄 ★  (키가 없으면 ChatOllama(model="gemma3:1b"))
```

Colab 은 0절 셀이 키를 이미 환경 변수에 넣었으므로 `load_dotenv()` 가 필요 없습니다. **체인 줄(`prompt | llm | parser`)은 0줄 변경**입니다.

### 실습 1-② 같은 체인 · 과제 3종 × 모델 2종 — 그 아래 셀 (`code/swap_models.py`)

| 과제 | 확인 방법 |
|---|---|
| ① 설명 | 눈으로 — 품질 1~5 |
| ② 형식 (딱 3줄, `1. 이름 - 설명`) | 줄 수 · 번호 |
| ③ 추론 (여러 단계) | 마지막 줄 숫자 |

**관찰 포인트**

1. 바뀐 코드는 **`llm =` 한 줄** 뿐이다 — prompt · parser · invoke 는 그대로 ★
2. 두 모델이 교체 가능한 이유 = 같은 Runnable 규약(`invoke`)을 따르기 때문
3. **'4B로 충분한가'의 답은 과제가 정한다** — 설명 · 형식 · 추론에서 결과가 다르다 ★
4. `usage_metadata`(LangChain 이 통일한 토큰 수)와 `response_metadata`(공급자 원본)는 모양이 다르다
5. 로컬 첫 호출의 로딩 시간은 `load_duration` 으로 **빼고** 본다 — 한 번만 실행하면 된다
6. '첫 토큰까지'는 `invoke` 로 잴 수 없다 → 3교시 실습 4에서 채운다 ★

> ⚠️ `ChatAnthropic` 은 쓰지 않습니다. Anthropic API 키가 필요하며,
> Claude Code 구독 계정으로는 호출할 수 없습니다.
>
> 🔶 `OPENAI_API_KEY` 가 없으면 **로컬 2종 비교**(`SMALL_MODEL` vs `MODEL`)로 자동 전환됩니다. 코드 구조는 그대로입니다.
> 모델을 6번 부릅니다 — 기다리는 동안 과제마다 어느 모델이 통과할지 **먼저 예측**해 적어 두세요.

In [ ]:
# ── 실습 1-① 내 체인에서 한 줄 바꾸기 ★★ ─────────────────────
#    실습실: copy ..\week03\first_chain.py first_chain.py → llm = 줄만 고쳐 python first_chain.py
#    Colab : 3주차 first_chain.py 의 부품 3개를 그대로 옮겨 왔습니다.
import os
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_openai import ChatOpenAI      # ← 추가한 import (실습실은 load_dotenv 두 줄도 추가)

MODEL = os.environ["MODEL"]
OPENAI_MODEL = "gpt-4o-mini"                 # 🔶 수업 전날 공식 문서에서 확인해 확정할 것

# ── 부품 3개 — 3주차 그대로 ──────────────────────────────
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 프로그래밍 강사입니다. {level} 눈높이로 설명하세요."),
        ("human", "{topic}의 장점 3가지를 각각 한 문장으로 알려줘."),
    ]
)
# llm = ChatOllama(model=MODEL)              # ← 3주차의 그 줄
if os.getenv("OPENAI_API_KEY"):
    llm = ChatOpenAI(model=OPENAI_MODEL)     # ← 바꾼 한 줄 ★
else:
    llm = ChatOllama(model=os.environ["SMALL_MODEL"])  # 키가 없으면 작은 로컬 모델 (실습실: ChatOllama(model="gemma3:1b"))
parser = StrOutputParser()

chain = prompt | llm | parser                # ← 체인 줄은 0줄 변경

print(f"[llm] {type(llm).__name__}")
print(chain.invoke({"level": "초보자", "topic": "파이썬"}).strip())
print()
print("[확인] 바꾼 것은 llm = 한 줄(과 import)뿐입니다. prompt · parser · chain · invoke 는 그대로입니다.")

In [ ]:
# ── 실습 1-② 같은 체인 · 과제 3종 × 모델 2종 — 정답 확인 · 속도 · 비용 ★★ ──
#    code/swap_models.py 와 같은 코드 (실습실: python swap_models.py — 모델 6번 호출)
#
#    chain = prompt | llm        ← llm 자리만 모델마다 다르다
#    (parser 는 응답 정보를 보려고 뒤에서 따로 적용합니다)
import os, re, time
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_openai import ChatOpenAI

# ── ① 고칠 수 있는 값은 모두 여기에 ─────────────────────────
LOCAL_MODEL = os.environ["MODEL"]
LOCAL_SMALL = os.environ["SMALL_MODEL"]      # 상용 키를 못 쓸 때의 대체 비교용

# 🔶 상용 모델명은 자주 바뀝니다. 수업 전날 공식 문서에서 확인해 확정할 것 ★
OPENAI_MODEL = "gpt-4o-mini"

# 최대 출력 토큰 — 뜻은 같은데 이름이 다릅니다 (Ollama: num_predict / OpenAI: max_tokens · 2절)
MAX_OUT = 400

# 🔶 단가 — 수업일 공식 가격표의 '100만 토큰당 달러' 를 그대로 옮겨 적는다 (아래는 연습용 가정값)
PRICE_IN_PER_1M = 0.15
PRICE_OUT_PER_1M = 0.60
KRW_PER_USD = 1400  # 🔶 수업일 환율

# 한 달 비용 시나리오 — 내 미니 프로젝트를 상상해서 고쳐 본다
USERS_PER_DAY = 100
CALLS_PER_USER = 10
DAYS_PER_MONTH = 30

# ── ② 프롬프트와 파서는 한 번만 만든다. 끝까지 바뀌지 않는다 ──
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 프로그래밍 강사입니다. {level} 눈높이로 설명하세요."),
        ("human", "{question}"),
    ]
)
parser = StrOutputParser()

# ── ③ 과제 3종 — 쉬운 것부터 ───────────────────────────────
TASKS = [
    {
        "id": "①", "name": "설명", "check": "눈으로",
        # 3주차 first_chain.py 와 같은 질문 — 3교시 streaming.py 의 숫자와 이어진다
        "question": "파이썬의 장점 3가지를 각각 한 문장으로 알려줘.",
    },
    {
        "id": "②", "name": "형식", "check": "3줄",
        "question": "파이썬 자료형 3가지를 '1. 이름 - 한 줄 설명' 형식으로 딱 3줄만 쓰세요. "
                    "인사말, 빈 줄, 다른 문장은 쓰지 마세요.",
    },
    {
        "id": "③", "name": "추론", "check": "숫자", "answer": 11,  # 🔶 OpenAI 쪽은 수업 전 공용 키로 1회 확인
        "question": "5명이 서로 한 번씩 악수했다. 그 뒤 그중 2명이 서로 한 번 더 악수했다. "
                    "악수는 모두 몇 번인가? "
                    "풀이를 쓰고 마지막 줄에 숫자만 써라.",
    },
]


def check_answer(task: dict, text: str) -> str:
    """정답 확인 — 눈으로 해도 되는 일을 대신 세어 줄 뿐, 채점기가 아닙니다."""
    lines = [line.strip() for line in text.strip().splitlines() if line.strip()]

    if task["check"] == "3줄":
        ok = len(lines) == 3 and all(line.startswith(f"{i}.") for i, line in enumerate(lines, 1))
        return f"{'통과' if ok else '실패'} (줄 {len(lines)}개)"

    if task["check"] == "숫자":
        nums = re.findall(r"\d+", lines[-1].replace(",", "")) if lines else []  # 마지막 줄의 숫자
        got = int(nums[-1]) if nums else None
        return f"{'통과' if got == task['answer'] else '실패'} (마지막 줄 {got} / 정답 {task['answer']})"

    return "눈으로 (품질 1~5)"


def build_models() -> tuple:
    """바뀌는 것은 이 목록뿐입니다. (모델 목록, 상용 키가 있는가) 를 돌려준다."""
    if os.getenv("OPENAI_API_KEY"):
        return {
            f"로컬 {LOCAL_MODEL}": ChatOllama(model=LOCAL_MODEL, temperature=0.2, num_predict=MAX_OUT),
            f"OpenAI {OPENAI_MODEL}": ChatOpenAI(model=OPENAI_MODEL, temperature=0.2, max_tokens=MAX_OUT),
        }, True

    # ── 대체안: 상용 키가 없으면 로컬 2종으로 비교한다 ──
    #    코드 구조는 완전히 같습니다. 비용은 '(가정) 상용 단가였다면' 으로만 계산하고,
    #    비교표의 비용 칸은 'ollama ps 의 SIZE(메모리 점유)' 로 바꿔 채웁니다.  ★
    print("[주의] OPENAI_API_KEY 가 없습니다 → 로컬 2종 비교로 진행합니다.")
    print("   (Colab 좌측 🔑 보안 비밀에 키를 넣고 0절 셀을 다시 실행하면 로컬 ↔ OpenAI 비교가 됩니다)")
    print()
    return {
        f"소형 {LOCAL_SMALL}": ChatOllama(model=LOCAL_SMALL, temperature=0.2, num_predict=MAX_OUT),
        f"중형 {LOCAL_MODEL}": ChatOllama(model=LOCAL_MODEL, temperature=0.2, num_predict=MAX_OUT),
    }, False


def read_speed(msg, elapsed: float) -> dict:
    """속도 분해 — 공급자마다 원본 정보(response_metadata)의 모양이 다릅니다."""
    meta = msg.response_metadata
    output_tokens = (msg.usage_metadata or {}).get("output_tokens")

    if "eval_duration" in meta:
        # Ollama: 구간별 시간을 나노초로 알려 준다 (3주차 2교시에서 읽어 본 그 값)
        load = meta.get("load_duration", 0) / 1e9
        gen = meta["eval_duration"] / 1e9
        return {
            "load": load,
            "net": elapsed - load,  # 로딩을 뺀 시간 ★
            "tps": meta.get("eval_count", 0) / gen if gen else None,  # 생성 구간만의 토큰/초
            "approx": False,
            "end": meta.get("done_reason"),
        }

    # OpenAI: 구간 정보가 없다 → 전체 시간으로 나눈 근사값 (네트워크 · 대기 포함이라 낮게 나온다)
    return {
        "load": None,
        "net": elapsed,
        "tps": output_tokens / elapsed if output_tokens and elapsed else None,
        "approx": True,
        "end": meta.get("finish_reason"),
    }


def cost_usd(input_tokens: int, output_tokens: int) -> tuple:
    """(입력 비용, 출력 비용) — 입력과 출력의 단가가 다릅니다."""
    return (
        input_tokens * PRICE_IN_PER_1M / 1_000_000,
        output_tokens * PRICE_OUT_PER_1M / 1_000_000,
    )


def print_month(input_tokens: int, output_tokens: int,
                users: int = USERS_PER_DAY, calls: int = CALLS_PER_USER) -> None:
    """1회 → 한 달. 곱하기로 봅니다."""
    c_in, c_out = cost_usd(input_tokens, output_tokens)
    once = c_in + c_out
    month_calls = users * calls * DAYS_PER_MONTH
    month = once * month_calls
    share = c_out / once * 100 if once else 0

    print(f"  단가          : 입력 ${PRICE_IN_PER_1M} · 출력 ${PRICE_OUT_PER_1M} (100만 토큰당)")
    print(f"  1회 호출      : 입력 {input_tokens} · 출력 {output_tokens} 토큰 → ${once:.6f} (약 {once * KRW_PER_USD:.2f}원)")
    print(f"  한 달 호출 수 : 하루 {users}명 × {calls}회 × {DAYS_PER_MONTH}일 = {month_calls:,}회")
    print(f"  한 달 비용    : ${month:,.2f} (약 {month * KRW_PER_USD:,.0f}원)")
    print(f"  출력 비중     : {share:.0f}%  ← 비용의 대부분이 출력 토큰에서 나오는가?")


def run_one(name: str, llm, task: dict, show_keys: bool) -> dict:
    """한 모델로 한 과제를 풀고 측정값을 돌려준다."""
    # 💡 파서를 빼면 AIMessage 가 그대로 온다 → 토큰 수와 원본 정보를 볼 수 있다
    #    (호출을 두 번 하지 않으려고 parser 는 뒤에서 따로 적용합니다.
    #     parser.invoke(msg) 가 곧 체인 끝의 parser 가 하던 일입니다.)
    chain = prompt | llm

    t0 = time.perf_counter()
    msg = chain.invoke({"level": "초보자", "question": task["question"]})
    elapsed = time.perf_counter() - t0

    text = parser.invoke(msg)
    usage = msg.usage_metadata or {}  # 🔶 공급자 · 버전에 따라 비어 있을 수 있음
    speed = read_speed(msg, elapsed)
    result = check_answer(task, text)

    print("=" * 64)
    print(f"[과제 {task['id']} {task['name']}] {name}")
    print("=" * 64)
    print(text.strip())
    print()
    print(f"  정답 확인      : {result}")
    print(f"  usage_metadata : {usage or '(측정 불가)'}")
    if show_keys:
        print(f"  response_metadata 키 : {sorted(msg.response_metadata)}")
    if speed["end"] == "length":
        print(f"  [잘림] 최대 출력 {MAX_OUT} 토큰에 걸렸습니다 — 그대로 기록하세요")
    print()

    return {
        "task": task,
        "name": name,
        "result": result,
        "elapsed": elapsed,
        "input_tokens": usage.get("input_tokens"),
        "output_tokens": usage.get("output_tokens"),
        **speed,
    }


def fmt(value, spec: str = ".1f") -> str:
    return "-" if value is None else format(value, spec)


def print_summary(rows: list, names: list, has_key: bool) -> None:
    """비교표(실습 2)에 옮겨 적을 수 있게 정리한다."""
    print("=" * 64)
    print("측정 요약 ① — 과제 × 모델")
    print("=" * 64)
    for r in rows:
        tps = ("≈" if r["approx"] else "") + fmt(r["tps"])
        print(f"  과제 {r['task']['id']} | {r['name']} | {r['result']}")
        print(f"      총 {r['elapsed']:.1f}초 · 로딩 {fmt(r['load'])}초 · 로딩 뺀 {r['net']:.1f}초"
              f" · {tps} 토큰/초 · 입력 {r['input_tokens']} · 출력 {r['output_tokens']}")

    print()
    print("=" * 64)
    print("측정 요약 ② — 비교표에 옮길 값")
    print("=" * 64)
    for name in names:
        mine = [r for r in rows if r["name"] == name]
        checked = [r for r in mine if r["task"]["check"] != "눈으로"]
        passed = sum(r["result"].startswith("통과") for r in checked)
        first = mine[0]  # 과제 ①
        speeds = [r["tps"] for r in mine if r["tps"]]
        avg_tps = sum(speeds) / len(speeds) if speeds else None
        is_local = first["load"] is not None

        print(f"[{name}]")
        print(f"  품질 · 정답 확인 통과      : {passed}/{len(checked)}  (과제 ① 품질 1~5 는 눈으로)")
        print(f"  속도 · 총 소요 (과제 ①)    : {first['net']:.1f}초  ← 로딩을 뺀 값")
        print(f"  속도 · 로딩 (과제 ①)       : {fmt(first['load']) + '초' if is_local else '해당 없음'}")
        print(f"  속도 · 생성 속도 (평균)    : {'≈' if first['approx'] else ''}{fmt(avg_tps)} 토큰/초")
        print(f"  속도 · 첫 토큰까지         : (3교시 실습 4)")
        print(f"  비용 · 입력/출력 (과제 ①)  : {first['input_tokens']} / {first['output_tokens']}")
        if is_local and has_key:
            print("  비용 · 1회 / 한 달         : 0원 (호출당 — 전기 · 장비 비용은 별도)")
        elif first["input_tokens"] is not None and first["output_tokens"] is not None:
            label = "(가정) 상용 단가였다면" if not has_key else "과제 ① 기준"
            print(f"  비용 · {label}")
            print_month(first["input_tokens"], first["output_tokens"])
        else:
            print("  비용                       : 측정 불가 (usage_metadata 없음)")
        print()

    print("[대기] '첫 토큰까지'는 invoke 로는 잴 수 없습니다 → 3교시 실습 4(stream)에서 채웁니다.  ★")
    print("[속도] 로컬은 load_duration 을 뺀 값입니다.")
    print("       OpenAI 의 토큰/초(≈)는 전체 시간으로 나눈 값이라 네트워크 시간이 섞여 낮게 나옵니다.")
    print("[비용] PRICE_* 가 수업일 단가인지 확인하세요. 내 시나리오로 다시 계산하려면:")
    print("       다음 셀 아래쪽의 MY_* 값을 바꿔 그 셀을 다시 실행 (실습실: python swap_models.py cost ...)")
    if not has_key:
        print("[대체] 키가 없어 로컬 2종을 비교했습니다. 비교표의 비용 칸은 '!ollama ps' SIZE 로 채우세요.")


# ── 실행 — 과제 → 모델 순서. 로컬 첫 호출(과제 ①)에 로딩이 들어간다 ──
models, HAS_KEY = build_models()
NAMES = list(models)
rows = []
for task in TASKS:
    for name, llm in models.items():
        rows.append(run_one(name, llm, task, show_keys=(task["id"] == "①")))

In [ ]:
# ── 측정 요약 — 실습 2 비교표에 옮겨 적으세요 ─────────────────
print_summary(rows, NAMES, HAS_KEY)

# ── 비용 — 1회에서 한 달로 ★ (모델을 부르지 않고 계산만) ──────
#    실습실: python swap_models.py cost 입력토큰 출력토큰 하루사용자 1인당호출
#    위 요약의 입력 · 출력 토큰과 내 미니 프로젝트 시나리오로 바꿔 이 셀을 다시 실행하세요.
MY_INPUT_TOKENS = 60
MY_OUTPUT_TOKENS = 150
MY_USERS_PER_DAY = 300
MY_CALLS_PER_USER = 5

print()
print("=" * 64)
print("비용 계산 — 모델을 부르지 않습니다")
print("=" * 64)
print_month(MY_INPUT_TOKENS, MY_OUTPUT_TOKENS, MY_USERS_PER_DAY, MY_CALLS_PER_USER)

### 실습 1-② 결과 읽기 ★

**① 정답 확인 — '4B로 충분한가'는 과제가 정합니다**

- 설명(①)처럼 단순한 과제는 로컬 4B로 충분한 경우가 많습니다. **형식(②)·추론(③)에서 결과가 갈리는지** 보십시오.
- ⚠️ '상용이 항상 좋다'로 끝내지 마십시오. **과제 난이도에 따라 답이 달라진다**는 것이 요점입니다.
- `check_answer()` 는 눈으로 해도 되는 일을 대신 세어 줄 뿐, **채점기가 아닙니다.** «실패» 인데 답이 맞아 보이면 마지막 줄·줄 수를 직접 확인하십시오.
- 한 번 돌린 결과는 **힌트이지 통계가 아닙니다** — 여러 번 재는 법은 5주차, 채점기를 만드는 법은 7주차.

**② 응답 정보 두 층**

| 층 | 무엇 | 공급자가 바뀌면 |
|---|---|---|
| `usage_metadata` | LangChain 이 통일한 토큰 수 (`input_tokens` · `output_tokens`) | 모양이 같다 — 🔶 공급자·버전에 따라 비면 «측정 불가» |
| `response_metadata` | 공급자 원본 | 다르다 — Ollama 는 `load_duration` · `eval_count` · `eval_duration` · `done_reason`, OpenAI 는 `finish_reason` (구간별 시간 없음) |

**③ 속도 분해 — 로딩 뺀 값**

| | 로컬 (Ollama) | OpenAI |
|---|---|---|
| 총 소요 (로딩 뺀 값) | 총 소요 − `load_duration` | 총 소요 그대로 |
| 로딩 | `load_duration` — 첫 호출(과제 ①)에서만 크다 | 해당 없음 |
| 토큰/초 | `eval_count ÷ eval_duration` (생성 구간만) | ≈ `output_tokens ÷ 총 소요` (네트워크 포함이라 낮게 나온다) |

> 나노초 → 초 환산은 3주차 2교시에서 했습니다. OpenAI 의 토큰/초(≈) 칸 하나로 결론 내지 마십시오.
> **[잘림]** 이 보이면 최대 출력 `MAX_OUT`(400 토큰)에 걸린 것입니다 — 그대로 기록합니다.

**④ 비용 — 1회에서 한 달로**

```
1회 비용   = (입력 토큰 × 입력 단가) + (출력 토큰 × 출력 단가)      ← 단가는 100만 토큰당
한 달 비용 = 1회 비용 × 하루 사용자 × 1인당 호출 × 30일
```

- 🔶 단가 · 환율은 위 셀 상단의 `PRICE_IN_PER_1M` · `PRICE_OUT_PER_1M` · `KRW_PER_USD` 입니다 (연습용 가정값 — 수업일 공식 가격표로 확정).
- 단가를 외우지 마십시오. 외울 것은 **계산 구조** — 출력이 대체로 더 비싸서 **출력 비중**이 크고, 비용은 1회가 아니라 **곱하기**로 봅니다.

## 실습 2 ★ (2교시) — 비교표 작성 (6기준)

`week04/비교표_양식.md` 를 채워 저장소에 커밋하십시오. 위 **«측정 요약 ② — 비교표에 옮길 값»** 을 옮겨 적습니다.

> 표시 — **[측정]** 스크립트가 잰 값 · **[계산]** 잰 값으로 계산한 값 · **[판단]** 재지 않았고 구조상 그렇다는 판단

**표 A — 과제별 정답 확인**

| 과제 | 확인 방법 | 로컬 (`gemma3:4b`) | 상용 (`gpt-4o-mini`) |
|---|---|---|---|
| ① 설명 | 눈으로 — 품질 1~5 **[판단]** | | |
| ② 형식 (딱 3줄) | 줄 수 · 번호 **[측정]** | | |
| ③ 추론 (여러 단계) | 마지막 줄 숫자 **[측정]** | | |

**표 B — 6기준**

| 기준 | 항목 | 로컬 (`gemma3:4b`) | 상용 (`gpt-4o-mini`) |
|---|---|---|---|
| 품질 | 정답 확인 ②·③ 통과 **[측정]** | /2 | /2 |
| 품질 | 응답 품질 ① (1~5) **[판단]** | | |
| 속도 | **총 소요 시간** (초 · 과제 ① · 로딩 뺀 값) **[측정]** | | |
| 속도 | └ 로딩 시간 (초 · 과제 ①) **[측정]** | | 해당 없음 |
| 속도 | 생성 속도 (토큰/초 · 평균) **[측정 · OpenAI 는 ≈]** | | ≈ |
| 속도 | **첫 토큰까지** (초) **[측정]** | ⏳ 3교시 실습 4에서 채움 | ⏳ |
| 비용 | 입력 / 출력 토큰 (과제 ①) **[측정]** | / | / |
| 비용 | 1회 비용 (과제 ①) **[계산]** | **0원** (호출당) | |
| 비용 | 한 달 비용 (하루 ___명 × ___회 × 30일) · 출력 비중 **[계산]** | **0원** (+ 장비 · 전기) | |
| 보안 | 데이터가 PC 밖으로 나가는가 **[판단 — 구조상]** | 나가지 않음 | 나감 (OpenAI 서버) |
| 오프라인 | 인터넷 없이 동작하는가 **[판단 — 구조상]** | 가능 | 불가 |
| VRAM | 필요 VRAM (1교시 계산) **[계산]** | ______ GB | 해당 없음 |

> 📌 비교표는 정규 과제가 아니지만 **버리지 마십시오.**
> **10주차 미니 프로젝트 명세**에서 *"왜 이 모델을 골랐는가"* 의 근거로 그대로 씁니다.
>
> ⚠️ 상용 키가 없으면 로컬 2종으로 채우고, **'비용' 행은 '메모리 점유'(`!ollama ps` 의 SIZE)** 로 치환합니다.
> (스크립트가 보여 주는 비용은 «(가정) 상용 단가였다면» — 참고로만 적으세요.)
>
> 상황별로 어떤 기준을 먼저 볼지는 **3교시 §4 «모델 선택 기준 정리»** 에서 다룹니다.

## 3교시 1절 — 공통 파라미터 4종

```
temperature · max_tokens      ← 모델에게 주는 지시
timeout · max_retries         ← 호출을 어떻게 할 것인가 (네트워크 계층)
```

뒤의 둘은 **로컬에서는 의미가 약합니다** — 인터넷을 안 타므로. 상용 API에서 중요합니다.

⚠️ `ChatOllama` 는 `max_tokens` · `timeout` · `max_retries` 를 **에러 없이 무시**합니다 (`max_tokens` 무시는 2교시 2절 ④에서 실측) — 로컬의 길이 제한은 `num_predict` 입니다.

In [ ]:
import os
from langchain_ollama import ChatOllama

MODEL    = os.environ["MODEL"]
QUESTION = "파이썬의 장점 3가지를 각각 한 문장으로 알려줘."

# ── ① temperature — 무작위성 ──
print("=" * 60)
print("① temperature - 무작위성")
print("=" * 60)

for temp in (0.0, 0.9):
    # num_predict 로 짧게 끊습니다 — 앞부분만 봐도 차이가 드러나고, 실습이 빨라집니다
    llm = ChatOllama(model=MODEL, temperature=temp, num_predict=60)
    print(f"\n── temperature={temp} : 같은 질문을 두 번 ──")
    for i in (1, 2):
        answer = llm.invoke(QUESTION).content.replace("\n", " ")
        print(f"  [{i}회] {answer[:70]}...")

print()
print("  0 ~ 0.3   : 분류·추출·요약·JSON 출력 - 일관성이 중요할 때 (5주차 구조화 출력)")
print("  0.7 ~ 1.0 : 아이디어 생성·창작")

In [ ]:
# ── ② 최대 생성 길이 ──
print("=" * 60)
print("② 최대 생성 길이 - Ollama: num_predict / OpenAI: max_tokens  [이름이 다름]")
print("=" * 60)

llm = ChatOllama(model=MODEL, temperature=0.2, num_predict=40)
print(llm.invoke(QUESTION).content)

print()
print("  [주의] '40토큰으로 요약해줘'가 아닙니다. 40토큰에서 잘립니다.")
print("     문장 중간에서 끊기죠? 길이를 조절하려면 프롬프트로도 함께 요청해야 합니다.")
print("     비용 상한을 거는 안전장치로 이해하는 것이 정확합니다.")

### ③ `timeout` / ④ `max_retries` — 호출을 어떻게 할 것인가

```python
ChatOpenAI(model="...", timeout=30)      # 30초 안에 응답 없으면 포기
ChatOpenAI(model="...", max_retries=2)   # 일시적 오류면 2번까지 다시 건다
```

> ⚠️ 둘 다 **상용 API(`ChatOpenAI`)용**입니다. `ChatOllama` 에 넣으면 **에러 없이 무시**됩니다.

```
    호출
      ├─ 성공 ─────────────────────▶ 끝
      └─ 실패 ─▶ max_retries 만큼 재시도
                    ├─ 성공 ────────▶ 끝
                    └─ 계속 실패 ─▶ 폴백(다른 모델)   ★ 실습 3
```

> **[주의]** 재시도가 듣는 것은 **'일시적' 오류뿐**입니다.
> 잘못된 키·없는 모델명은 몇 번을 걸어도 실패합니다. → 그때 필요한 것이 폴백.

**[추가] 재시도 간격 — 지수 백오프(Exponential Backoff)**

> 바로 다시 걸지 않고, 실패할 때마다 **기다리는 시간을 2배로** 늘립니다. 줄어드는 것이 아니라 **늘어납니다.**
> `0.5초 → 1초 → 2초 → 4초 → 8초(상한)` — 실제로는 0.75~1.0배로 조금씩 흔듭니다(**지터**).
> `max_retries` 만 쓰면 openai 라이브러리가 **이미** 해 줍니다. 대신 재시도가 많을수록 폴백이 늦어집니다.

## 실습 3 (3교시) — 상용 실패 → 로컬 대체 `with_fallbacks`

```
     상용 API 호출
          │
    실패(장애·한도 초과·키 문제)
          │
          ▼
  로컬 모델로 자동 전환      ← 서비스는 계속된다
```

**관찰 포인트**

1. 주 모델이 실패했는데 **에러 없이 결과가 나온다** ★
2. ⚠️ 그래서 장애가 **'조용히' 묻힙니다.**
   어떤 모델이 실제로 응답했는지 기록해야 합니다 → **6주차 LangSmith**

In [ ]:
import os
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_openai import ChatOpenAI

LOCAL_MODEL = os.environ["MODEL"]

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 프로그래밍 강사입니다. {level} 눈높이로 설명하세요."),
        ("human",  "{topic}의 장점 3가지를 각각 한 문장으로 알려줘."),
    ]
)
parser = StrOutputParser()
INPUTS = {"level": "초보자", "topic": "파이썬"}


def build_primary():
    """주 모델 — 일부러 실패하게 만든다 ★

    실패를 만드는 방법 세 가지
      ① 없는 모델명   model="gpt-존재하지-않는-모델"  ← 권장
      ② 잘못된 키     api_key="sk-틀린키"             ← 키를 망가뜨릴 위험
      ③ 극단적 타임아웃 timeout=0.001                 ← 재현이 들쭉날쭉
    """
    if os.getenv("OPENAI_API_KEY"):
        return "OpenAI(없는 모델명)", ChatOpenAI(
            model="gpt-존재하지-않는-모델",   # 실패 유발 ①
            timeout=10,
            max_retries=0,                    # 재시도 없이 바로 실패시켜 시간을 아낀다
        )

    # 키가 없어도 실습은 그대로 됩니다. 주 모델을 '없는 로컬 모델'로 바꾸면
    # 로컬 → 로컬 폴백이 되고, 관찰할 것은 완전히 같습니다.
    print("[주의] OPENAI_API_KEY 가 없습니다 → 주 모델을 '없는 로컬 모델'로 대체합니다.")
    print()
    return "로컬(없는 모델명)", ChatOllama(model="없는-모델-이름")


primary_name, primary = build_primary()

# ── 대체 모델: 로컬 ──
backup = ChatOllama(model=LOCAL_MODEL, temperature=0.2)

# ── 폴백 연결 ★ — 이 한 줄이 전부입니다 ──
llm = primary.with_fallbacks([backup])

print("=" * 60)
print(f"주 모델({primary_name})을 부릅니다 …")
print("=" * 60)

# 체인의 형태는 지금까지와 똑같습니다
#     chain = prompt | llm | parser
# 다만 여기서는 '누가 대답했는지'까지 보려고 파서를 빼고 호출합니다.
msg = (prompt | llm).invoke(INPUTS)
print(parser.invoke(msg))

# ── ⚠️ 폴백의 대가 — 누가 대답했는지 확인해 봅시다 ──────
meta   = msg.response_metadata or {}
actual = meta.get("model_name") or meta.get("model") or "(알 수 없음)"

print()
print("=" * 60)
print(f"실제로 응답한 모델: {actual}")
print("=" * 60)

| 폴백이 해결하는 것 | 해결하지 못하는 것 |
|---|---|
| 서비스가 멈추지 않는다 | 응답 품질이 떨어진다 (사용자는 모른 채 받는다) |
| 장애 대응 코드가 짧다 | 실패가 조용히 묻힌다 — 로그가 없으면 눈치 못 챔 |
| (없음) | 로컬 모델이 VRAM 에 안 올라가면 **폴백도 실패** ★ |

> **[주의]** 폴백이 걸려 있으면 장애가 안 보입니다.
> 그래서 '어떤 모델이 실제로 응답했는지'를 위처럼 기록해야 합니다.
> 이 기록을 자동으로 남겨 주는 도구가 **6주차 LangSmith** 입니다.

## 실습 4 ★ (3교시) — 첫 토큰까지의 시간(TTFT) 측정

```
[invoke]  질문 ──────────── (10초 침묵) ────────────▶ 답 전체가 한 번에
[stream]  질문 ─ 0.8초 ─▶ 답 ─ 이 ─ 조 ─ 금 ─ 씩 ─ 나 ─ 온 ─ 다 ▶ (총 10초)
```

**총 소요 시간은 같습니다. 달라지는 것은 '첫 글자가 언제 나오는가' 입니다.**

**관찰 포인트**

1. 바뀐 것은 `invoke` → `stream` 과 반복문뿐. 체인은 그대로다 ★
2. 총 시간 ≈ 변화 없음 → **스트리밍은 빨라지게 하지 않는다**
3. 첫 토큰까지 ≪ 총 시간 → 개선되는 것은 **'체감 속도'**
4. 여기서 잰 값으로 **2교시 비교표의 '첫 토큰까지' 칸을 채웁니다** ★★

> 🔶 로컬 모델이 메모리에서 내려가 있으면 첫 토큰까지가 크게 늘어납니다
> (모델 로딩 시간이 포함되므로). 바로 위 셀들을 먼저 실행해 모델을 올려 두십시오.

In [ ]:
import os, time
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_openai import ChatOpenAI

LOCAL_MODEL  = os.environ["MODEL"]
OPENAI_MODEL = "gpt-4o-mini"       # 🔶 수업 전날 공식 문서에서 확인해 확정할 것

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 프로그래밍 강사입니다. {level} 눈높이로 설명하세요."),
        ("human",  "{topic}의 장점 3가지를 각각 한 문장으로 알려줘."),
    ]
)
INPUTS = {"level": "초보자", "topic": "파이썬"}


def measure_invoke(chain) -> float:
    """① invoke — 다 나올 때까지 기다린다."""
    t0 = time.perf_counter()
    chain.invoke(INPUTS)
    total = time.perf_counter() - t0
    print(f"[invoke] 총 {total:.2f}초 (그동안 화면은 조용했습니다)")
    return total


def measure_stream(chain) -> tuple:
    """② stream — 오는 대로 출력한다. 첫 조각이 온 시각을 기록한다."""
    t0, first = time.perf_counter(), None

    for chunk in chain.stream(INPUTS):        # ★ invoke → stream, 이것뿐입니다
        if first is None and chunk:
            first = time.perf_counter() - t0  # 첫 글자가 도착한 시각
        print(chunk, end="", flush=True)

    total = time.perf_counter() - t0
    ttft  = f"{first:.2f}초" if first is not None else "측정 불가"
    print(f"\n[stream] 첫 토큰까지 {ttft} / 총 {total:.2f}초")
    return first, total


def run(name: str, llm) -> dict:
    chain = prompt | llm | StrOutputParser()
    print("=" * 60)
    print(f"[{name}]")
    print("=" * 60)
    invoke_total = measure_invoke(chain)
    print()
    ttft, stream_total = measure_stream(chain)
    print()
    return {"name": name, "invoke_total": invoke_total,
            "ttft": ttft, "stream_total": stream_total}


models = {f"로컬 {LOCAL_MODEL}": ChatOllama(model=LOCAL_MODEL, temperature=0.2)}
if os.getenv("OPENAI_API_KEY"):
    models[f"OpenAI {OPENAI_MODEL}"] = ChatOpenAI(model=OPENAI_MODEL, temperature=0.2)
else:
    print("[주의] OPENAI_API_KEY 가 없어 로컬 모델만 측정합니다.\n")

srows = [run(name, llm) for name, llm in models.items()]

In [ ]:
print("=" * 60)
print("비교표의 '첫 토큰까지' 칸을 지금 채우세요  ★")
print("=" * 60)
print(f"  {'모델':<24} {'invoke 총':>10} {'stream 총':>10} {'첫 토큰까지':>12}")
for r in srows:
    ttft = f"{r['ttft']:.2f}초" if r["ttft"] is not None else "측정불가"
    print(f"  {r['name']:<24} {r['invoke_total']:>9.2f}초 "
          f"{r['stream_total']:>9.2f}초 {ttft:>12}")

print()
print("  총 시간 ≒ 변화 없음      → 스트리밍은 빨라지게 하지 않는다")
print("  첫 토큰까지 ≪ 총 시간    → 개선되는 것은 '체감 속도'")
print()
print("[정리] 스트리밍은 성능 최적화가 아니라 '사용자 경험' 개선입니다.")
print("       총 시간이 같아도 사용자는 훨씬 빠르다고 느낍니다.")

> **비동기 버전 (소개만)**
>
> ```python
> async for chunk in chain.astream(INPUTS):
>     print(chunk, end="", flush=True)
> ```
>
> 웹 서버처럼 여러 요청을 동시에 받아야 할 때 씁니다.
> 본 교과목에서는 직접 쓰지 않습니다. '이런 게 있다' 정도로 넘어갑니다.

## 오늘 확인할 것

- [ ] `ollama show` 로 파라미터 수·컨텍스트를 조회하고 **8GB 기준**으로 판정했다
- [ ] `ollama ps` 의 `SIZE`·`PROCESSOR` 를 계산값과 대조했다
- [ ] Modelfile 로 **나만의 챗봇**을 만들고, 코드 설정이 이기는 것을 **토큰 수**로 확인했다
- [ ] `.bind()` 를 틀려 보고, `max_tokens` 가 **조용히 무시**되는 것을 출력으로 확인했다
- [ ] 3주차 체인에서 **`llm =` 한 줄 교체**로 모델을 바꿨다 ★★
- [ ] 과제 3종 × 모델 2종을 측정하고 **한 달 비용**을 계산했다 ★
- [ ] `with_fallbacks` 로 폴백이 걸리는 것을 확인했다
- [ ] `stream()` 으로 **첫 토큰까지**를 측정했다 ★
- [ ] **비교표 6기준**을 완성해 저장소에 커밋했다

> ⚠️ **Colab 값을 그대로 비교표에 적지 마십시오.**
> VRAM 판정은 **실습실 8GB 기준**, 속도는 **런타임이 GPU였는지 함께 기록**하십시오.